# Campanhas persistentes de modelos

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="05_treinamento_campanha"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'hi.demo.notebook.a9291c01',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': True,
 'confirmation': 'EXECUTE_DEMO',
 'request_key': 'campaign-a9291c01',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'create',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': True,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Catálogo e orçamento. Chaves hi.pt.demanda representam períodos cadastrados DEMO; não são fontes institucionais. Default: três modelos, máximo de dois processos.

In [2]:
display(pd.DataFrame(call(cfg,"GET","/v1/training/campaigns")))
from src.data.journey import extract
_,snapshot=extract(cfg,snapshot_id=str(params.snapshot_id) if params.snapshot_id else None)
display(snapshot)
scopes=call(cfg,"GET","/v1/audit/scopes")
display(pd.DataFrame(scopes))
if params.model_count>len(scopes):
    raise ValueError("O catálogo DEMO não possui esse número de séries distintas; use até "+str(len(scopes)))
model_keys=[s["model_key"] for s in scopes[:params.model_count]]
display({"model_count":len(model_keys),"max_parallel_jobs":params.parallel_jobs,"first_keys":model_keys[:5]})


,campaign_id,status,requested_count,created_at_utc,correlation_id
0,4c559b77-6699-4fe2-9ade-dce8ecd33c90,COMPLETED,1,2026-11-04T21:43:05.656881+00:00,3f1c5325-3afc-40b5-bded-0207ebb993c7
1,6fa95eaf-75a5-423d-b6bd-cf19612fb001,COMPLETED,1,2026-11-04T21:42:46.908485+00:00,8c7176ee-c952-48ef-be57-3adef09528c9
2,272115ed-4d91-48ba-b863-5aabd49dbf0a,COMPLETED,1,2026-11-04T21:41:47.472453+00:00,75ab8e5b-7ed4-4e24-93de-326f3c890304
3,2ecb5a7f-4d8d-48e9-bfc4-cbafc347c736,COMPLETED,1,2026-11-04T21:41:38.403663+00:00,7b34d4d5-b855-4add-b9c7-076dd5c2ec67
4,645ca303-e2d0-4490-bcf6-6b43bacfc6d1,COMPLETED,3,2026-11-04T21:41:33.217738+00:00,48d1c356-9676-45ce-afff-4802ce3c8bdc
5,2a9e2e23-0173-4d19-83f9-129ce49affff,COMPLETED,1,2026-11-04T21:26:02.529098+00:00,18cb82de-428b-47df-bcf1-94cb9200b337


{'id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
 'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
 'target_index': 4052,
 'target_semester': 1,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'query_sha256': 'defc10d1b6ea752600af271285c54107663e91db9df13914aea7e42613c83adf',
 'query_version': 'journey-history-v1'}

,catalog_id,campus_id,course_id,period_id,campus_code,course_code,period_number,period_code,model_family,model_key,model_display_name
0,c8454d20-98fc-5cf4-91bd-8e88b6c3c209,44bf155f-0e52-5044-bf25-e2cf4ca73019,49386655-1d9f-5160-a775-8605d1fd0d5b,ca85135b-5d7f-5b48-9607-d21bef7b19e7,DEMO-C1,DEMO-CR1,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...
1,89021437-58a6-5166-9899-086fe6f333b1,44bf155f-0e52-5044-bf25-e2cf4ca73019,49386655-1d9f-5160-a775-8605d1fd0d5b,d857c32c-2ede-57b0-8f4b-c518c6f58b0c,DEMO-C1,DEMO-CR1,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR1.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...
2,871f4e77-eafd-5fb4-a094-9419851165a2,44bf155f-0e52-5044-bf25-e2cf4ca73019,6208faec-f9b8-5784-82ab-bbbb46bd8d28,c4119e4d-2581-55b4-8247-851618d67d24,DEMO-C1,DEMO-CR2,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR2.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR2 • Pe...
3,d4acd4b1-99c5-5f6d-a2a6-1ca872717462,44bf155f-0e52-5044-bf25-e2cf4ca73019,6208faec-f9b8-5784-82ab-bbbb46bd8d28,5dbb1a48-f1c2-5ede-81d4-d9ce7822f847,DEMO-C1,DEMO-CR2,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR2.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR2 • Pe...
4,b2fc44f8-5fe3-5bd1-b373-0a8af04e7d6e,44bf155f-0e52-5044-bf25-e2cf4ca73019,a5739734-976b-561d-b4c3-acbb5ed5cf83,b30529c9-d516-5198-9cee-d437f659e7ff,DEMO-C1,DEMO-CR3,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR3.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR3 • Pe...
5,b43f2bcc-74ce-5c54-91ee-8cd32eba1290,44bf155f-0e52-5044-bf25-e2cf4ca73019,a5739734-976b-561d-b4c3-acbb5ed5cf83,003a4014-8321-5d09-b9bc-0f04d766cc80,DEMO-C1,DEMO-CR3,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR3.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR3 • Pe...
6,01d4a060-7041-598e-a863-2e7a9114fe9e,44bf155f-0e52-5044-bf25-e2cf4ca73019,710ed550-edf1-5b07-baff-f2484c029fd7,1242140c-7db4-52dd-97c1-8d59049bc649,DEMO-C1,DEMO-CR4,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR4.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR4 • Pe...
7,7f3d893e-681a-5281-9a82-7fe4d7ce65c9,44bf155f-0e52-5044-bf25-e2cf4ca73019,710ed550-edf1-5b07-baff-f2484c029fd7,156d48f4-518d-56a5-979d-583889c73b00,DEMO-C1,DEMO-CR4,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR4.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR4 • Pe...
8,8080b7a3-2bd6-5acd-8899-0f9d95df3847,44bf155f-0e52-5044-bf25-e2cf4ca73019,09786e25-3a0e-5ebe-805c-207a72d98b74,de0c4144-8a18-5c60-8573-743cb74009ce,DEMO-C1,DEMO-CR5,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR5.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR5 • Pe...
9,75ad2cd2-e90b-50ba-bbbe-ecb868862ab2,44bf155f-0e52-5044-bf25-e2cf4ca73019,09786e25-3a0e-5ebe-805c-207a72d98b74,92cbdefd-c378-5e89-b349-680d8e40f22a,DEMO-C1,DEMO-CR5,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR5.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR5 • Pe...


{'model_count': 30,
 'max_parallel_jobs': 2,
 'first_keys': ['hi.pt.demanda.DEMO-C1.DEMO-CR1.P1',
  'hi.pt.demanda.DEMO-C1.DEMO-CR1.P2',
  'hi.pt.demanda.DEMO-C1.DEMO-CR2.P1',
  'hi.pt.demanda.DEMO-C1.DEMO-CR2.P2',
  'hi.pt.demanda.DEMO-C1.DEMO-CR3.P1']}

2. Controle opt-in: pause/resume/cancel/retry-failed. Cancelamento é cooperativo; itens em execução concluem, pendentes são cancelados.

In [3]:
if params.campaign_action not in ("create","observe"):
    params.authorize()
    if not params.campaign_id: raise ValueError("campaign_id obrigatório")
    evidence["control"]=call(cfg,"POST",f"/v1/training/campaigns/{params.campaign_id}/{params.campaign_action}")
    display(evidence["control"])


3. Criar campanha somente quando campaign_action=create, ou observar o ID escolhido. Estado/checkpoints permanecem no PostgreSQL após fechar o notebook.

In [4]:
if params.campaign_action!="create":
    params=params.model_copy(update={"execute_write":False})
from src.workbench.operations_sdk import campaign
if params.execute_write:
    params.authorize()
    if not params.snapshot_id:
        raise ValueError("Selecione snapshot_id da tabela SQL")
    payload=dict(snapshot_id=str(params.snapshot_id),cycle_code=cfg.cycle,model_keys=model_keys,
        algorithm=params.algorithm,alpha=params.alpha,max_parallel_jobs=params.parallel_jobs,
        requested_by=params.requester,confirm_training=True,campaign_display_name=params.campaign_display_name)
    display({"preflight":payload,"budget":"máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas"})
    submitted=campaign(cfg,payload,params.request_key)
    campaign_id=submitted["campaign_id"]
else:
    campaign_id=str(params.campaign_id) if params.campaign_id else ""
if campaign_id:
    state=await_job(cfg,f"/v1/training/campaigns/{campaign_id}",timeout=7200)
    items=call(cfg,"GET",f"/v1/training/campaigns/{campaign_id}/items?limit=500")
    display(state);display(pd.DataFrame(items))
    evidence.update(campaign=state,items=items)
    if params.execute_write and state["status"]!="COMPLETED":
        report(NOTEBOOK_ID,evidence,status="FAIL",generation="v3")
        raise RuntimeError("Campanha não completou: "+state["status"]+"; consulte os erros dos jobs persistidos")
else:
    evidence["training"]="NOT_EXECUTED: escolha campanha existente ou habilite escrita"


{'preflight': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'cycle_code': '2026-1',
  'model_keys': ['hi.pt.demanda.DEMO-C1.DEMO-CR1.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR1.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR2.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR2.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR3.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR3.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR4.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR4.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR5.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR5.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR1.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR1.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR2.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR2.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR3.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR3.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR4.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR4.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR5.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR5.P2',
   'hi.pt.demanda.DEMO-C3.DEMO-CR1.P1',
   'hi.pt.demanda.DEMO-C3.DEMO-CR1.P2',
   'hi.pt.

{'status': 'RUNNING', 'counts': {'PENDING': 28, 'RUNNING': 2}, 'utc': '2026-11-04T21:43:26.259465+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 2, 'PENDING': 26, 'RUNNING': 2}, 'utc': '2026-11-04T21:43:28.278585+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 4, 'PENDING': 24, 'RUNNING': 2}, 'utc': '2026-11-04T21:43:30.296631+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 6, 'PENDING': 22, 'RUNNING': 2}, 'utc': '2026-11-04T21:43:32.316748+00:00'}


{'status': 'RUNNING', 'counts': {'COMPLETED': 10, 'PENDING': 18, 'RUNNING': 2}, 'utc': '2026-11-04T21:43:34.334948+00:00'}


... (6 outputs left out)

{'status': 'RUNNING', 'counts': {'COMPLETED': 29, 'RUNNING': 1}, 'utc': '2026-11-04T21:43:48.457517+00:00'}


{'status': 'COMPLETED', 'counts': {'COMPLETED': 30}, 'utc': '2026-11-04T21:43:50.487429+00:00'}


{'campaign_id': '1e148711-47b3-4750-9346-f4df35def14e',
 'idempotency_key': 'campaign-a9291c01',
 'request_hash': '4c180b93d644852a6b6a90145f1ff4e491a00349d323d1b75e8b912c937cb60f',
 'request': {'alpha': 1.0,
  'algorithm': 'ridge',
  'cycle_code': '2026-1',
  'model_keys': ['hi.pt.demanda.DEMO-C1.DEMO-CR1.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR1.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR2.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR2.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR3.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR3.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR4.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR4.P2',
   'hi.pt.demanda.DEMO-C1.DEMO-CR5.P1',
   'hi.pt.demanda.DEMO-C1.DEMO-CR5.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR1.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR1.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR2.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR2.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR3.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR3.P2',
   'hi.pt.demanda.DEMO-C2.DEMO-CR4.P1',
   'hi.pt.demanda.DEMO-C2.DEMO-CR4.P2',
   'hi.

,job_id,campaign_id,model_key,status,attempt,heartbeat_at_utc,started_at_utc,ended_at_utc,mlflow_run_id,registered_model_name,model_version,error_code,result
0,85cd45cb-c45d-4ef2-9200-72cf41678188,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,COMPLETED,1,2026-11-04T21:43:40.159553+00:00,2026-11-04T21:43:39.254344+00:00,2026-11-04T21:43:40.532211+00:00,19495cf828a946d2924858e554ce5641,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,3,None,"{'run_id': '19495cf828a946d2924858e554ce5641',..."
1,3bd1d167-127f-41fb-9539-ebbbe59e3ca6,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR1.P2,COMPLETED,1,2026-11-04T21:43:30.733358+00:00,2026-11-04T21:43:29.691801+00:00,2026-11-04T21:43:31.135700+00:00,2e6951af59584fc59e50376ff4c3dc99,hi.pt.demanda.DEMO-C1.DEMO-CR1.P2,2,None,"{'run_id': '2e6951af59584fc59e50376ff4c3dc99',..."
2,947f2053-44f2-488f-a323-0b1dc9d1ac27,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR2.P1,COMPLETED,1,2026-11-04T21:43:40.174314+00:00,2026-11-04T21:43:39.256828+00:00,2026-11-04T21:43:40.530010+00:00,62f5daa090c44ec096d0af03840d0269,hi.pt.demanda.DEMO-C1.DEMO-CR2.P1,2,None,"{'run_id': '62f5daa090c44ec096d0af03840d0269',..."
3,0a303ff2-1c7d-419b-af95-39e5d9084492,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR2.P2,COMPLETED,1,2026-11-04T21:43:29.222572+00:00,2026-11-04T21:43:28.091839+00:00,2026-11-04T21:43:29.661051+00:00,a80e9b36e4764129b935ff4ff804fc71,hi.pt.demanda.DEMO-C1.DEMO-CR2.P2,1,None,"{'run_id': 'a80e9b36e4764129b935ff4ff804fc71',..."
4,ef9291f1-4ef6-472d-a817-ed58fe59a234,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR3.P1,COMPLETED,1,2026-11-04T21:43:47.992799+00:00,2026-11-04T21:43:47.064187+00:00,2026-11-04T21:43:48.361987+00:00,e767ef652f9944c99c802bbe5439b24d,hi.pt.demanda.DEMO-C1.DEMO-CR3.P1,1,None,"{'run_id': 'e767ef652f9944c99c802bbe5439b24d',..."
5,6ee87f81-caba-4ed5-ba19-234dc0be26a8,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR3.P2,COMPLETED,1,2026-11-04T21:43:37.376990+00:00,2026-11-04T21:43:36.256337+00:00,2026-11-04T21:43:37.869323+00:00,4720a0478b5e4fb8a222f34cec636060,hi.pt.demanda.DEMO-C1.DEMO-CR3.P2,1,None,"{'run_id': '4720a0478b5e4fb8a222f34cec636060',..."
6,71c8af6a-ff22-4dc3-911b-6c5a8a9c9094,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR4.P1,COMPLETED,1,2026-11-04T21:43:37.381896+00:00,2026-11-04T21:43:36.258692+00:00,2026-11-04T21:43:37.871710+00:00,8df95e2a01644d8b8d51d94dbee29fb1,hi.pt.demanda.DEMO-C1.DEMO-CR4.P1,1,None,"{'run_id': '8df95e2a01644d8b8d51d94dbee29fb1',..."
7,083e3107-a855-4ac1-8178-0a3aed20bb0f,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR4.P2,COMPLETED,1,2026-11-04T21:43:27.560400+00:00,2026-11-04T21:43:26.231226+00:00,2026-11-04T21:43:28.060951+00:00,85495fb0f42b401985587bd3b77e02ce,hi.pt.demanda.DEMO-C1.DEMO-CR4.P2,1,None,"{'run_id': '85495fb0f42b401985587bd3b77e02ce',..."
8,ee5003be-83d4-496c-8216-bce76e908b09,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR5.P1,COMPLETED,1,2026-11-04T21:43:46.992352+00:00,2026-11-04T21:43:45.815928+00:00,2026-11-04T21:43:47.417036+00:00,24571ecb25404ceab73644ba7df4cd0e,hi.pt.demanda.DEMO-C1.DEMO-CR5.P1,1,None,"{'run_id': '24571ecb25404ceab73644ba7df4cd0e',..."
9,7853cb4c-39aa-46a4-8ef7-3ecd71f92f6e,1e148711-47b3-4750-9346-f4df35def14e,hi.pt.demanda.DEMO-C1.DEMO-CR5.P2,COMPLETED,1,2026-11-04T21:43:38.793132+00:00,2026-11-04T21:43:37.895714+00:00,2026-11-04T21:43:39.224696+00:00,6a45925fe11e4172bd8497db80286b97,hi.pt.demanda.DEMO-C1.DEMO-CR5.P2,1,None,"{'run_id': '6a45925fe11e4172bd8497db80286b97',..."


4. Quantidades comprovadas e progresso. Treinos reais, versões e instâncias de serving são números distintos.

In [5]:
items=evidence.get("items",[])
display({"requested":len(items),"completed":sum(i["status"]=="COMPLETED" for i in items),"registered_versions":sum(bool(i["model_version"]) for i in items),"automatically_promoted":0,"new_HTTP_servers":0})
display_locations(cfg,items)


{'requested': 30,
 'completed': 30,
 'registered_versions': 30,
 'automatically_promoted': 0,
 'new_HTTP_servers': 0}

### Onde conferir

,destination,label,reference
0,Modelo DEMO,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1/3
1,Experimento,HI/POC/PlanilhaTurmas/Demanda/2026-1,http://127.0.0.1:5000/#/experiments/3
2,Run,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1__ridge__2026...,http://127.0.0.1:5000/#/experiments/3/runs/194...
3,Registry,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1 • versão 3,http://127.0.0.1:5000/#/models/hi.pt.demanda.D...
4,RustFS,Console • catálogo e arquivos reais,http://localhost:9001
5,Manifest,Índice imutável HI-LINEAGE/1.1,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...
6,Arquivo RustFS,MLmodel,s3://pucpr-poc/mlflow/3/models/m-66f067f12cd14...
7,Arquivo RustFS,model.pkl,s3://pucpr-poc/mlflow/3/models/m-66f067f12cd14...
8,Arquivo RustFS,requirements.txt,s3://pucpr-poc/mlflow/3/models/m-66f067f12cd14...
9,PostgreSQL,planejamento_db.planning.v_model_inventory,SELECT * FROM planning.v_model_inventory WHERE...


[Experimento](http://127.0.0.1:5000/#/experiments/3)

[Run](http://127.0.0.1:5000/#/experiments/3/runs/19495cf828a946d2924858e554ce5641)

[Registry](http://127.0.0.1:5000/#/models/hi.pt.demanda.DEMO-C1.DEMO-CR1.P1/versions/3)

... (202 outputs left out)

[Registry](http://127.0.0.1:5000/#/models/hi.pt.demanda.DEMO-C3.DEMO-CR5.P2/versions/1)

[RustFS](http://localhost:9001)

[API GET Bearer](http://localhost:8000/v1/audit/model?model_key=hi.pt.demanda.DEMO-C3.DEMO-CR5.P2&version=1)

[Dashboard](http://localhost:8088)

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [6]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/05_treinamento_campanha.json'